In [29]:
import polars as pl

edges = pl.read_csv("datasets/edges.csv")

edges.head()

osm_id,from_id,to_id,distance_m,fclass,oneway,maxspeed
i64,i64,i64,f64,str,i64,i64
23262895,0,1,841.23,"""residential""",0,0
23262948,2,3,611.05,"""residential""",0,0
23262982,4,5,489.21,"""residential""",0,0
23262989,6,7,713.1,"""residential""",0,20
23263232,8,9,254.74,"""tertiary""",0,0


In [30]:
print(edges.filter(pl.col("distance_m") < 0))


shape: (0, 7)
┌────────┬─────────┬───────┬────────────┬────────┬────────┬──────────┐
│ osm_id ┆ from_id ┆ to_id ┆ distance_m ┆ fclass ┆ oneway ┆ maxspeed │
│ ---    ┆ ---     ┆ ---   ┆ ---        ┆ ---    ┆ ---    ┆ ---      │
│ i64    ┆ i64     ┆ i64   ┆ f64        ┆ str    ┆ i64    ┆ i64      │
╞════════╪═════════╪═══════╪════════════╪════════╪════════╪══════════╡
└────────┴─────────┴───────┴────────────┴────────┴────────┴──────────┘


In [31]:
print(edges.filter(pl.col("maxspeed") < 0))

shape: (0, 7)
┌────────┬─────────┬───────┬────────────┬────────┬────────┬──────────┐
│ osm_id ┆ from_id ┆ to_id ┆ distance_m ┆ fclass ┆ oneway ┆ maxspeed │
│ ---    ┆ ---     ┆ ---   ┆ ---        ┆ ---    ┆ ---    ┆ ---      │
│ i64    ┆ i64     ┆ i64   ┆ f64        ┆ str    ┆ i64    ┆ i64      │
╞════════╪═════════╪═══════╪════════════╪════════╪════════╪══════════╡
└────────┴─────────┴───────┴────────────┴────────┴────────┴──────────┘


In [32]:
dup_edges = edges.group_by("osm_id").agg(pl.len().alias("cantidad")).filter(pl.col("cantidad") > 1).get_column("osm_id")
dup_edges


osm_id
i64


Como al hacer el checkeo de Datos no se encontro ningun negativo, no se hizo ninguna accion de eliminacion.

In [33]:
max_speed_by_fclass = edges.group_by("fclass").agg(
    pl.col("maxspeed").mean().round(0).cast(pl.Int64).alias("maxspeed_media"),
    pl.col("maxspeed").median().round(0).cast(pl.Int64).alias("maxspeed_mediana"),
    pl.col("maxspeed").mode().first().alias("maxspeed_moda"),
)

max_speed_by_fclass

fclass,maxspeed_media,maxspeed_mediana,maxspeed_moda
str,i64,i64,i64
"""service""",0,0,0
"""unclassified""",7,0,0
"""motorway""",80,80,80
"""track""",0,0,0
"""trunk_link""",10,0,0
…,…,…,…
"""track_grade2""",8,0,0
"""trunk""",57,80,80
"""track_grade3""",7,0,0


In [34]:
fclass = (edges.select("fclass").unique().to_series().to_list())
print(fclass)

['tertiary_link', 'unknown', 'pedestrian', 'secondary', 'residential', 'unclassified', 'secondary_link', 'trunk', 'primary_link', 'track_grade2', 'living_street', 'motorway', 'track_grade5', 'service', 'primary', 'track_grade4', 'busway', 'footway', 'path', 'motorway_link', 'bridleway', 'track_grade1', 'trunk_link', 'track', 'steps', 'cycleway', 'tertiary', 'track_grade3']


In [35]:
speed_map = {
    "motorway": 80,
    "motorway_link": 80,
    "trunk": 80,
    "trunk_link": 80,
    "primary": 80,
    "primary_link": 80,
    "secondary": 70,
    "secondary_link": 70,
    "tertiary": 70,
    "tertiary_link": 70,
    "residential": 40,
    "living_street": 20,
    "service": 20,
    "pedestrian": 10,
    "busway": 40,
    "footway": 10,
    "cycleway": 10,
    "path": 10,
    "steps": 5,
    "bridleway": 10,
    "track": 70,
    "track_grade1": 70,
    "track_grade2": 60,
    "track_grade3": 50,
    "track_grade4": 40,
    "track_grade5": 30,
    "unclassified": 40,
    "unknown": 40,
}

In [38]:
df = edges.with_columns(
    pl.when(pl.col("maxspeed") == 0)
    .then(
        pl.col("fclass").replace(speed_map)
    )
    .otherwise(pl.col("maxspeed"))
    .alias("maxspeed")
)
df

ComputeError: cannot compare string with numeric type (i32)

This error occurred in the following expression:
	[(col("maxspeed")) == (dyn int: 0)]
while evaluating this larger expression:
	.when([(col("maxspeed")) == (dyn int: 0)]).then(col("fclass").replace([["motorway", "motorway_link", … "unknown"], [80, 80, … 40]])).otherwise(col("maxspeed"))
